SECTION 1 Setup

(Import libraries)

In [ ]:
!pip install scikit-surprise -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 16.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.metrics import mean_squared_error

from surprise import Dataset
from surprise import Reader
from surprise import SVD
from surprise import SVDpp

SECTION 2 Load Split

(Load the fixed train-test split)

In [ ]:
train_df = pd.read_csv(
    "train_per_user_temporal.csv"
)

test_df = pd.read_csv(
    "test_per_user_temporal.csv"
)

print(train_df.shape)
print(test_df.shape)

(1506935, 4)
(493065, 4)


SECTION 3 RMSE Function

(Evaluation metric)

In [ ]:
def rmse(actual, predicted):

    return np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

SECTION 4 Build Surprise Dataset

(Prepare data for training)

In [ ]:
reader = Reader(
    rating_scale=(1,5)
)

surprise_data = Dataset.load_from_df(
    train_df[
        [
            "user_id",
            "movie_id",
            "rating"
        ]
    ],
    reader
)

trainset = (
    surprise_data
    .build_full_trainset()
)

SECTION 5 Baseline SVD

(Use best tuned SVD from Notebook 08)

In [ ]:
svd_model = SVD(
    n_factors=100,
    n_epochs=50,
    lr_all=0.003,
    reg_all=0.05,
    random_state=42
)

In [ ]:
svd_model.fit(
    trainset
)

In [ ]:
svd_predictions = []

for row in tqdm(
    test_df.itertuples(),
    total=len(test_df)
):

    pred = svd_model.predict(
        row.user_id,
        row.movie_id
    )

    svd_predictions.append(
        pred.est
    )

100%|██████████| 493065/493065 [00:05<00:00, 92000.44it/s] 


In [ ]:
svd_rmse = rmse(
    test_df["rating"],
    svd_predictions
)

print(
    "SVD RMSE:",
    round(
        svd_rmse,
        6
    )
)

SVD RMSE: 0.970288


SECTION 6 SVD++

(Incorporating implicit feedback)

In [ ]:
svdpp_model = SVDpp(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

In [ ]:
svdpp_model.fit(
    trainset
)

In [ ]:
svdpp_predictions = []

for row in tqdm(
    test_df.itertuples(),
    total=len(test_df)
):

    pred = svdpp_model.predict(
        row.user_id,
        row.movie_id
    )

    svdpp_predictions.append(
        pred.est
    )

100%|██████████| 493065/493065 [00:17<00:00, 27816.60it/s]


In [ ]:
svdpp_rmse = rmse(
    test_df["rating"],
    svdpp_predictions
)

print(
    "SVD++ RMSE:",
    round(
        svdpp_rmse,
        6
    )
)

SVD++ RMSE: 0.978777


SECTION 8 Model Comparison

(Compare matrix factorization approaches)

In [ ]:
comparison = pd.DataFrame({

    "Model":[
        "SVD",
        "SVD++"
    ],

    "RMSE":[
        svd_rmse,
        svdpp_rmse
    ]
})

comparison.sort_values(
    "RMSE"
)

,Model,RMSE
0,SVD,0.970288
1,SVD++,0.978777
